# 模型选择

本 Notebook 用于：

1. 在 **开发集 `df_dev.xlsx`** 上对 6 个候选模型进行模型选择  
   候选模型：`GBR`、`XGBR`、`ETR`、`RFR`、`KNNR`、`DNNR_rand`

2. **第一阶段不再使用 Optuna**  
   直接调用“文件3（缺失值插补阶段）”中已经得到的各目标最优超参数，进行 **Repeated K-Fold CV** 比较  
   这样可以显著降低总耗时

3. 选出每个目标的冠军模型后，进入 **第二阶段 Optuna 正式调参**

4. 用 **全开发集** 训练最终模型，并在 **独立测试集 `df_test.xlsx`** 上完成最终验证


In [1]:
# ============================================================
# Cell 1：导入依赖与全局参数
# ============================================================
import os
import json
import math
import copy
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, RepeatedKFold

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor
from sklearn.neighbors import KNeighborsRegressor

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

try:
    from xgboost import XGBRegressor
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False
    XGBRegressor = None

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

warnings.filterwarnings('ignore')

# ---------------------------
# 路径配置
# ---------------------------
DEV_PATH = Path('df_dev.xlsx')
TEST_PATH = Path('df_test.xlsx')
OUTPUT_DIR = Path('model_selection')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------
# 特征与目标
# ---------------------------
FEATURES = [
    'Si', 'Fe', 'Cu', 'Mn', 'Mg', 'Cr', 'Zn', 'V', 'Ti', 'Zr',
    'Ni', 'Be', 'Sc', 'Tsol', 'Tage', 'tage'
]
TARGETS = ['YS', 'UTS', 'El']

# ---------------------------
# 随机种子
# ---------------------------
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ---------------------------
# 第一阶段：仅做固定参数比较
# ---------------------------
N_SPLITS_STAGE1 = 5
N_REPEATS_STAGE1 = 5

# ---------------------------
# 第二阶段：冠军模型 Optuna 调参
# 需要更快时可以把 FINAL_N_TRIALS 调小到 20 或 30
# ---------------------------
FINAL_CV_SPLITS = 5
FINAL_N_TRIALS = 20

print('当前设备:', DEVICE)
print('XGBoost 可用:', XGB_AVAILABLE)
print('输出目录:', OUTPUT_DIR.resolve())


当前设备: cuda
XGBoost 可用: True
输出目录: E:\al_alloy_fast\model_selection


In [2]:
# ============================================================
# Cell 2：读取数据并做基础检查
# ============================================================
assert DEV_PATH.exists(), f'未找到开发集文件: {DEV_PATH}'
assert TEST_PATH.exists(), f'未找到测试集文件: {TEST_PATH}'

df_dev = pd.read_excel(DEV_PATH)
df_test = pd.read_excel(TEST_PATH)

print('df_dev shape :', df_dev.shape)
print('df_test shape:', df_test.shape)

missing_in_dev = [c for c in FEATURES + TARGETS if c not in df_dev.columns]
missing_in_test = [c for c in FEATURES + TARGETS if c not in df_test.columns]

assert len(missing_in_dev) == 0, f'开发集缺少列: {missing_in_dev}'
assert len(missing_in_test) == 0, f'测试集缺少列: {missing_in_test}'

print('\n开发集目标缺失情况:')
display(df_dev[TARGETS].isnull().sum().to_frame('missing_count'))

print('\n测试集目标缺失情况:')
display(df_test[TARGETS].isnull().sum().to_frame('missing_count'))


df_dev shape : (256, 22)
df_test shape: (64, 22)

开发集目标缺失情况:


,missing_count
YS,0
UTS,0
El,0



测试集目标缺失情况:


,missing_count
YS,0
UTS,0
El,0


In [3]:
# ============================================================
# Cell 3：文件3中已经得到的固定超参数
# 这些参数直接来自你上传的“缺失值插补 notebook”的输出结果
# 第一阶段将直接使用这些参数，不再重复 Optuna
# ============================================================
STAGE1_FIXED_PARAMS = {
    'YS': {
        'GBR': {
            'learning_rate': 0.08216962745870568,
            'loss': 'absolute_error',
            'n_estimators': 157,
            'max_depth': 14,
            'alpha': 0.5687311407272482
        },
        'XGBR': {
            'learning_rate': 0.0885391461883862,
            'n_estimators': 194,
            'max_depth': 49,
            'subsample': 0.33370111001086733,
            'colsample_bytree': 0.8625649342047428
        },
        'ETR': {
            'n_estimators': 497,
            'max_depth': 20
        },
        'RFR': {
            'n_estimators': 35,
            'max_depth': 29
        },
        'KNNR': {
            'algorithm': 'ball_tree',
            'n_neighbors': 1,
            'leaf_size': 36,
            'p': 3
        },
        'DNNR_rand': {
            'n_layers': 6,
            'activation': 'relu',
            'lr': 0.0014580768310414134,
            'optimizer': 'Adam',
            'batch_size': 24,
            'units_0': 128,
            'units_1': 160,
            'units_2': 128,
            'units_3': 224,
            'units_4': 224,
            'units_5': 192
        }
    },
    'UTS': {
        'GBR': {
            'learning_rate': 0.10769622478263129,
            'loss': 'absolute_error',
            'n_estimators': 191,
            'max_depth': 20,
            'alpha': 0.8022294011541319
        },
        'XGBR': {
            'learning_rate': 0.12279231304548048,
            'n_estimators': 129,
            'max_depth': 41,
            'subsample': 0.4236388708355149,
            'colsample_bytree': 0.9983007503043428
        },
        'ETR': {
            'n_estimators': 240,
            'max_depth': 16
        },
        'RFR': {
            'n_estimators': 209,
            'max_depth': 43
        },
        'KNNR': {
            'algorithm': 'kd_tree',
            'n_neighbors': 2,
            'leaf_size': 38,
            'p': 3
        },
        'DNNR_rand': {
            'n_layers': 5,
            'activation': 'relu',
            'lr': 0.003073842725448067,
            'optimizer': 'Adam',
            'batch_size': 24,
            'units_0': 128,
            'units_1': 96,
            'units_2': 64,
            'units_3': 128,
            'units_4': 96
        }
    },
    'El': {
        'GBR': {
            'learning_rate': 0.10121098823604023,
            'loss': 'absolute_error',
            'n_estimators': 183,
            'max_depth': 19,
            'alpha': 0.6847316467588064
        },
        'XGBR': {
            'learning_rate': 0.03664778904535752,
            'n_estimators': 197,
            'max_depth': 7,
            'subsample': 0.7253165649772657,
            'colsample_bytree': 0.32419174389200633
        },
        'ETR': {
            'n_estimators': 403,
            'max_depth': 40
        },
        'RFR': {
            'n_estimators': 88,
            'max_depth': 45
        },
        'KNNR': {
            'algorithm': 'kd_tree',
            'n_neighbors': 2,
            'leaf_size': 38,
            'p': 3
        },
        'DNNR_rand': {
            'n_layers': 3,
            'activation': 'tanh',
            'lr': 0.0027947231216553206,
            'optimizer': 'RMSprop',
            'batch_size': 32,
            'units_0': 160,
            'units_1': 96,
            'units_2': 96
        }
    }
}

print('第一阶段固定超参数字典已加载。')
for t in TARGETS:
    print(t, '->', list(STAGE1_FIXED_PARAMS[t].keys()))


第一阶段固定超参数字典已加载。
YS -> ['GBR', 'XGBR', 'ETR', 'RFR', 'KNNR', 'DNNR_rand']
UTS -> ['GBR', 'XGBR', 'ETR', 'RFR', 'KNNR', 'DNNR_rand']
El -> ['GBR', 'XGBR', 'ETR', 'RFR', 'KNNR', 'DNNR_rand']


In [4]:
# ============================================================
# Cell 4：通用工具函数
# ============================================================
def calc_reg_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    return {
        'R2': float(r2_score(y_true, y_pred)),
        'MAE': float(mean_absolute_error(y_true, y_pred)),
        'RMSE': float(np.sqrt(mean_squared_error(y_true, y_pred))),
    }

def safe_std(x):
    x = np.asarray(x, dtype=float)
    if len(x) <= 1:
        return 0.0
    return float(np.std(x, ddof=1))

def make_output_subdir(name):
    p = OUTPUT_DIR / name
    p.mkdir(parents=True, exist_ok=True)
    return p


In [5]:
# ============================================================
# Cell 5：定义 DNNR_rand 对应的 sklearn 风格封装
# 注意：
# 1. 这里保留 DNNR_rand，不删除
# 2. 支持文件3里的随机层结构参数
# 3. 也支持第二阶段 Optuna 搜索
# ============================================================
class _DNNNet(nn.Module):
    def __init__(self, input_dim, hidden_layers, activation='relu', dropout=0.0):
        super().__init__()
        act_map = {
            'relu': nn.ReLU,
            'gelu': nn.GELU,
            'elu': nn.ELU,
            'tanh': nn.Tanh,
            'sigmoid': nn.Sigmoid,
            'leaky_relu': nn.LeakyReLU
        }
        act_cls = act_map.get(activation, nn.ReLU)

        layers = []
        in_dim = input_dim
        for h in hidden_layers:
            layers.append(nn.Linear(in_dim, h))
            if activation == 'leaky_relu':
                layers.append(act_cls(0.1))
            else:
                layers.append(act_cls())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

class DNNRegressorRand(BaseEstimator, RegressorMixin):
    def __init__(
        self,
        input_dim=None,
        hidden_layers=(64, 32),
        activation='relu',
        optimizer_name='Adam',
        lr=1e-3,
        batch_size=32,
        epochs=200,
        patience=20,
        dropout=0.0,
        weight_decay=0.0,
        random_state=42,
        verbose=False,
        use_gpu=True
    ):
        self.input_dim = input_dim
        self.hidden_layers = hidden_layers
        self.activation = activation
        self.optimizer_name = optimizer_name
        self.lr = lr
        self.batch_size = batch_size
        self.epochs = epochs
        self.patience = patience
        self.dropout = dropout
        self.weight_decay = weight_decay
        self.random_state = random_state
        self.verbose = verbose
        self.use_gpu = use_gpu

    def _set_seed(self):
        np.random.seed(self.random_state)
        random.seed(self.random_state)
        torch.manual_seed(self.random_state)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(self.random_state)

    def fit(self, X, y):
        self._set_seed()

        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.float32).reshape(-1, 1)

        self.scaler_ = StandardScaler()
        Xs = self.scaler_.fit_transform(X).astype(np.float32)

        device = torch.device('cuda' if (self.use_gpu and torch.cuda.is_available()) else 'cpu')

        self.model_ = _DNNNet(
            input_dim=self.input_dim if self.input_dim is not None else X.shape[1],
            hidden_layers=self.hidden_layers,
            activation=self.activation,
            dropout=self.dropout
        ).to(device)

        opt_map = {
            'Adam': torch.optim.Adam,
            'SGD': torch.optim.SGD,
            'RMSprop': torch.optim.RMSprop
        }
        opt_cls = opt_map.get(self.optimizer_name, torch.optim.Adam)

        optimizer = opt_cls(
            self.model_.parameters(),
            lr=self.lr,
            weight_decay=self.weight_decay
        )
        criterion = nn.MSELoss()

        Xt = torch.tensor(Xs, dtype=torch.float32)
        yt = torch.tensor(y, dtype=torch.float32)

        dataset = TensorDataset(Xt, yt)
        loader = DataLoader(dataset, batch_size=min(self.batch_size, len(dataset)), shuffle=True)

        best_loss = np.inf
        best_state = None
        no_improve = 0

        for epoch in range(self.epochs):
            self.model_.train()
            epoch_losses = []

            for xb, yb in loader:
                xb = xb.to(device)
                yb = yb.to(device)

                optimizer.zero_grad()
                pred = self.model_(xb)
                loss = criterion(pred, yb)
                loss.backward()
                optimizer.step()

                epoch_losses.append(loss.item())

            mean_loss = float(np.mean(epoch_losses)) if epoch_losses else np.inf

            if mean_loss < best_loss - 1e-8:
                best_loss = mean_loss
                best_state = {k: v.detach().cpu().clone() for k, v in self.model_.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1

            if self.verbose and ((epoch + 1) % 20 == 0 or epoch == 0):
                print(f'Epoch {epoch+1:03d} | loss={mean_loss:.6f}')

            if no_improve >= self.patience:
                break

        if best_state is not None:
            self.model_.load_state_dict(best_state)

        self.model_.eval()
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=np.float32)
        Xs = self.scaler_.transform(X).astype(np.float32)

        device = torch.device('cuda' if (self.use_gpu and torch.cuda.is_available()) else 'cpu')
        self.model_.to(device)
        self.model_.eval()

        with torch.no_grad():
            pred = self.model_(torch.tensor(Xs, dtype=torch.float32).to(device)).cpu().numpy().ravel()
        return pred


In [6]:
# ============================================================
# Cell 6：模型构建函数
# 第一阶段和第二阶段都使用这一套接口
# ============================================================
def convert_fixed_dnn_params(params):
    n_layers = params['n_layers']
    hidden_layers = tuple(params[f'units_{i}'] for i in range(n_layers))
    return {
        'hidden_layers': hidden_layers,
        'activation': params['activation'],
        'optimizer_name': params.get('optimizer', 'Adam'),
        'lr': params['lr'],
        'batch_size': params['batch_size'],
        'epochs': params.get('epochs', 200),
        'patience': params.get('patience', 20),
        'dropout': params.get('dropout', 0.0),
        'weight_decay': params.get('weight_decay', 0.0),
    }

def build_model(model_name, params, input_dim=None, for_search=False):
    if model_name == 'GBR':
        return GradientBoostingRegressor(
            random_state=RANDOM_SEED,
            **params
        )

    if model_name == 'XGBR':
        if not XGB_AVAILABLE:
            raise RuntimeError('当前环境未安装 xgboost，无法使用 XGBR。')

        xgb_params = dict(params)
        xgb_params.update({
            'random_state': RANDOM_SEED,
            'verbosity': 0,
            'n_jobs': -1,
            'tree_method': 'hist'
        })

        # 搜索阶段默认用 CPU 更稳，最终训练可用 GPU
        if (not for_search) and torch.cuda.is_available():
            xgb_params['device'] = 'cuda'
        else:
            xgb_params['device'] = 'cpu'

        return XGBRegressor(**xgb_params)

    if model_name == 'ETR':
        return ExtraTreesRegressor(
            random_state=RANDOM_SEED,
            n_jobs=-1,
            **params
        )

    if model_name == 'RFR':
        return RandomForestRegressor(
            random_state=RANDOM_SEED,
            n_jobs=-1,
            **params
        )

    if model_name == 'KNNR':
        return Pipeline([
            ('scaler', StandardScaler()),
            ('model', KNeighborsRegressor(**params))
        ])

    if model_name == 'DNNR_rand':
        dnn_params = params if 'hidden_layers' in params else convert_fixed_dnn_params(params)
        return DNNRegressorRand(
            input_dim=input_dim,
            random_state=RANDOM_SEED,
            verbose=False,
            use_gpu=True,
            **dnn_params
        )

    raise ValueError(f'不支持的模型名称: {model_name}')

# 快速测试
_ = build_model('GBR', STAGE1_FIXED_PARAMS['YS']['GBR'], input_dim=len(FEATURES), for_search=True)
print('build_model 已就绪。')


build_model 已就绪。


In [7]:
# ============================================================
# Cell 7：第一阶段 —— 直接使用文件3已有超参数做 Repeated CV
# 不再进行 Optuna，因此这一阶段会快很多
# ============================================================
candidate_models = ['GBR', 'XGBR', 'ETR', 'RFR', 'KNNR', 'DNNR_rand']

stage1_records = []

for target in TARGETS:
    print(f'\n{"="*12} Stage 1 | Target: {target} {"="*12}')

    df_dev_t = df_dev.dropna(subset=[target]).reset_index(drop=True)
    X_all = df_dev_t[FEATURES].values
    y_all = df_dev_t[target].values

    rkf = RepeatedKFold(
        n_splits=N_SPLITS_STAGE1,
        n_repeats=N_REPEATS_STAGE1,
        random_state=RANDOM_SEED
    )

    for model_name in candidate_models:
        params = copy.deepcopy(STAGE1_FIXED_PARAMS[target][model_name])

        split_mae, split_rmse, split_r2 = [], [], []

        for split_id, (tr_idx, va_idx) in enumerate(rkf.split(X_all), start=1):
            X_tr, X_va = X_all[tr_idx], X_all[va_idx]
            y_tr, y_va = y_all[tr_idx], y_all[va_idx]

            model = build_model(
                model_name=model_name,
                params=params,
                input_dim=X_tr.shape[1],
                for_search=False
            )
            model.fit(X_tr, y_tr)
            pred = model.predict(X_va)

            m = calc_reg_metrics(y_va, pred)
            split_mae.append(m['MAE'])
            split_rmse.append(m['RMSE'])
            split_r2.append(m['R2'])

        rec = {
            'target': target,
            'model': model_name,
            'MAE_mean': float(np.mean(split_mae)),
            'MAE_std': safe_std(split_mae),
            'RMSE_mean': float(np.mean(split_rmse)),
            'RMSE_std': safe_std(split_rmse),
            'R2_mean': float(np.mean(split_r2)),
            'R2_std': safe_std(split_r2)
        }
        stage1_records.append(rec)

        print(
            f'[{target}] {model_name} | '
            f'MAE={rec["MAE_mean"]:.4f}±{rec["MAE_std"]:.4f} | '
            f'RMSE={rec["RMSE_mean"]:.4f}±{rec["RMSE_std"]:.4f} | '
            f'R2={rec["R2_mean"]:.4f}±{rec["R2_std"]:.4f}'
        )

stage1_results_df = pd.DataFrame(stage1_records).sort_values(
    by=['target', 'R2_mean', 'MAE_mean'],
    ascending=[True, False, True]
).reset_index(drop=True)

stage1_dir = make_output_subdir('stage1_fixed_param_screening')
stage1_results_df.to_excel(stage1_dir / 'stage1_fixed_param_screening_results.xlsx', index=False)

print('\n第一阶段完成。')
display(stage1_results_df)



============ Stage 1 | Target: YS ============
[YS] GBR | MAE=26.7490±3.1651 | RMSE=38.2823±5.0207 | R2=0.8817±0.0332
[YS] XGBR | MAE=28.5650±3.3742 | RMSE=40.6131±4.7485 | R2=0.8676±0.0310
[YS] ETR | MAE=26.5944±3.4681 | RMSE=39.2223±4.9754 | R2=0.8761±0.0327
[YS] RFR | MAE=28.8638±3.5919 | RMSE=40.9940±5.6273 | R2=0.8646±0.0371
[YS] KNNR | MAE=35.1627±4.6562 | RMSE=49.9165±6.7261 | R2=0.7997±0.0500
[YS] DNNR_rand | MAE=36.6284±7.5528 | RMSE=54.9520±15.3288 | R2=0.7429±0.1659

============ Stage 1 | Target: UTS ============
[UTS] GBR | MAE=23.6027±4.0569 | RMSE=33.5882±6.1202 | R2=0.8865±0.0446
[UTS] XGBR | MAE=23.9660±2.9752 | RMSE=34.5127±4.6059 | R2=0.8816±0.0369
[UTS] ETR | MAE=22.9880±3.0625 | RMSE=33.7285±4.6143 | R2=0.8870±0.0350
[UTS] RFR | MAE=25.4591±3.3189 | RMSE=35.0012±4.3698 | R2=0.8784±0.0358
[UTS] KNNR | MAE=28.4277±4.0683 | RMSE=40.0027±4.6687 | R2=0.8422±0.0419
[UTS] DNNR_rand | MAE=34.5436±6.5645 | RMSE=52.6255±13.1716 | R2=0.7116±0.1616

============ Stage 1 | Tar

,target,model,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_mean,R2_std
0,El,GBR,1.952309,0.259583,3.138839,0.656685,0.737566,0.079117
1,El,XGBR,2.014074,0.307655,3.191114,0.670627,0.729599,0.083077
2,El,ETR,1.917375,0.299350,3.206454,0.728835,0.727554,0.084959
3,El,RFR,2.044319,0.317273,3.228528,0.701989,0.721482,0.092863
4,El,DNNR_rand,2.266773,0.389813,3.706587,0.776068,0.636547,0.102021
5,El,KNNR,2.461617,0.450964,3.759745,0.744998,0.621048,0.120426
6,UTS,ETR,22.988017,3.062466,33.728491,4.614327,0.887018,0.035022
7,UTS,GBR,23.602736,4.056890,33.588242,6.120185,0.886517,0.044600
8,UTS,XGBR,23.965962,2.975240,34.512672,4.605874,0.881559,0.036894
9,UTS,RFR,25.459058,3.318938,35.001206,4.369844,0.878388,0.035849


In [8]:
# ============================================================
# Cell 8：按 R2 优先、MAE 次优 选择冠军模型
# ============================================================
selected_rows = []

for target in TARGETS:
    sub = stage1_results_df[stage1_results_df['target'] == target].copy()
    sub = sub.sort_values(by=['R2_mean', 'MAE_mean'], ascending=[False, True]).reset_index(drop=True)
    selected_rows.append(sub.iloc[0])

champion_df = pd.DataFrame(selected_rows).reset_index(drop=True)
champion_df = champion_df.rename(columns={'model': 'selected_model'})

stage1_champions = dict(zip(champion_df['target'], champion_df['selected_model']))

print('各目标第一阶段冠军模型：')
display(champion_df)

champion_df.to_excel(stage1_dir / 'stage1_champion_models.xlsx', index=False)


各目标第一阶段冠军模型：


,target,selected_model,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_mean,R2_std
0,YS,GBR,26.749017,3.165094,38.282275,5.020689,0.881697,0.033158
1,UTS,ETR,22.988017,3.062466,33.728491,4.614327,0.887018,0.035022
2,El,GBR,1.952309,0.259583,3.138839,0.656685,0.737566,0.079117


In [9]:
# ============================================================
# Cell 9：第二阶段搜索空间
# 这里只对冠军模型做 Optuna
# ============================================================
def suggest_params(model_name, trial):
    if model_name == 'GBR':
        return {
            'n_estimators': trial.suggest_int('n_estimators', 80, 300, step=10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.20, log=True),
            'max_depth': trial.suggest_int('max_depth', 2, 8),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
            'loss': trial.suggest_categorical('loss', ['squared_error', 'absolute_error', 'huber']),
            'alpha': trial.suggest_float('alpha', 0.1, 0.95)
        }

    if model_name == 'XGBR':
        if not XGB_AVAILABLE:
            raise RuntimeError('当前环境未安装 xgboost，无法使用 XGBR。')
        return {
            'n_estimators': trial.suggest_int('n_estimators', 80, 300, step=10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.20, log=True),
            'max_depth': trial.suggest_int('max_depth', 2, 12),
            'subsample': trial.suggest_float('subsample', 0.4, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0),
            'min_child_weight': trial.suggest_float('min_child_weight', 1, 10),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-6, 1.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-6, 10.0, log=True)
        }

    if model_name == 'ETR':
        return {
            'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=20),
            'max_depth': trial.suggest_int('max_depth', 3, 30),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
            'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
            'bootstrap': trial.suggest_categorical('bootstrap', [False, True])
        }

    if model_name == 'RFR':
        return {
            'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=20),
            'max_depth': trial.suggest_int('max_depth', 3, 30),
            'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
            'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
            'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
            'bootstrap': trial.suggest_categorical('bootstrap', [True, False])
        }

    if model_name == 'KNNR':
        return {
            'n_neighbors': trial.suggest_int('n_neighbors', 1, 20),
            'weights': trial.suggest_categorical('weights', ['uniform', 'distance']),
            'algorithm': trial.suggest_categorical('algorithm', ['auto', 'ball_tree', 'kd_tree', 'brute']),
            'leaf_size': trial.suggest_int('leaf_size', 10, 50),
            'p': trial.suggest_int('p', 1, 5)
        }

    if model_name == 'DNNR_rand':
        n_layers = trial.suggest_int('n_layers', 2, 6)
        hidden_layers = tuple(
            trial.suggest_int(f'units_{i}', 32, 256, step=32) for i in range(n_layers)
        )
        return {
            'hidden_layers': hidden_layers,
            'activation': trial.suggest_categorical('activation', ['relu', 'tanh', 'gelu', 'elu']),
            'optimizer_name': trial.suggest_categorical('optimizer_name', ['Adam', 'RMSprop', 'SGD']),
            'lr': trial.suggest_float('lr', 1e-4, 5e-3, log=True),
            'batch_size': trial.suggest_categorical('batch_size', [16, 24, 32, 48, 64]),
            'epochs': trial.suggest_int('epochs', 120, 240, step=20),
            'patience': trial.suggest_int('patience', 10, 30),
            'dropout': trial.suggest_float('dropout', 0.0, 0.3),
            'weight_decay': trial.suggest_float('weight_decay', 1e-7, 1e-2, log=True)
        }

    raise ValueError(f'不支持的模型名称: {model_name}')


In [10]:
# ============================================================
# Cell 10：第二阶段 —— 仅对冠军模型做 Optuna
# 这里保留 enqueue_trial，把文件3中的参数作为一个初始候选解
# ============================================================
def convert_fixed_params_for_enqueue(model_name, params):
    params = copy.deepcopy(params)

    if model_name == 'DNNR_rand':
        # DNN 的第二阶段搜索空间使用 hidden_layers 风格
        return None

    return params

stage2_records = []
final_best_params = {}

for target in TARGETS:
    champion_model = stage1_champions[target]
    print(f'\n{"="*12} Stage 2 | Target: {target} | Champion: {champion_model} {"="*12}')

    df_dev_t = df_dev.dropna(subset=[target]).reset_index(drop=True)
    X_all = df_dev_t[FEATURES].values
    y_all = df_dev_t[target].values

    cv = KFold(n_splits=FINAL_CV_SPLITS, shuffle=True, random_state=RANDOM_SEED)

    def objective(trial):
        params = suggest_params(champion_model, trial)

        fold_mae = []
        for tr_idx, va_idx in cv.split(X_all):
            X_tr, X_va = X_all[tr_idx], X_all[va_idx]
            y_tr, y_va = y_all[tr_idx], y_all[va_idx]

            model = build_model(
                model_name=champion_model,
                params=params,
                input_dim=X_tr.shape[1],
                for_search=True
            )
            model.fit(X_tr, y_tr)
            pred = model.predict(X_va)
            fold_mae.append(mean_absolute_error(y_va, pred))

        return float(np.mean(fold_mae))

    study = optuna.create_study(direction='minimize')

    # 先把文件3里的固定参数作为一个初始解塞进去
    fixed_params = convert_fixed_params_for_enqueue(champion_model, STAGE1_FIXED_PARAMS[target][champion_model])
    if fixed_params is not None:
        try:
            study.enqueue_trial(fixed_params)
        except Exception:
            pass

    study.optimize(objective, n_trials=FINAL_N_TRIALS, show_progress_bar=False)

    best_params = study.best_trial.params
    final_best_params[target] = best_params

    # 用最佳参数重新统计 5-fold CV 指标
    cv_mae, cv_rmse, cv_r2 = [], [], []
    for tr_idx, va_idx in cv.split(X_all):
        X_tr, X_va = X_all[tr_idx], X_all[va_idx]
        y_tr, y_va = y_all[tr_idx], y_all[va_idx]

        model = build_model(
            model_name=champion_model,
            params=best_params,
            input_dim=X_tr.shape[1],
            for_search=False
        )
        model.fit(X_tr, y_tr)
        pred = model.predict(X_va)

        m = calc_reg_metrics(y_va, pred)
        cv_mae.append(m['MAE'])
        cv_rmse.append(m['RMSE'])
        cv_r2.append(m['R2'])

    rec = {
        'target': target,
        'selected_model': champion_model,
        'CV_MAE_mean': float(np.mean(cv_mae)),
        'CV_MAE_std': safe_std(cv_mae),
        'CV_RMSE_mean': float(np.mean(cv_rmse)),
        'CV_RMSE_std': safe_std(cv_rmse),
        'CV_R2_mean': float(np.mean(cv_r2)),
        'CV_R2_std': safe_std(cv_r2),
        'best_value_MAE': float(study.best_value)
    }
    stage2_records.append(rec)

    print(
        f'[{target}] {champion_model} | '
        f'CV_MAE={rec["CV_MAE_mean"]:.4f}±{rec["CV_MAE_std"]:.4f} | '
        f'CV_RMSE={rec["CV_RMSE_mean"]:.4f}±{rec["CV_RMSE_std"]:.4f} | '
        f'CV_R2={rec["CV_R2_mean"]:.4f}±{rec["CV_R2_std"]:.4f}'
    )

stage2_results_df = pd.DataFrame(stage2_records)
stage2_dir = make_output_subdir('stage2_final_tuning')
stage2_results_df.to_excel(stage2_dir / 'stage2_final_tuning_results.xlsx', index=False)

with open(stage2_dir / 'final_best_params.json', 'w', encoding='utf-8') as f:
    json.dump(final_best_params, f, ensure_ascii=False, indent=2)

print('\n第二阶段完成。')
display(stage2_results_df)



============ Stage 2 | Target: YS | Champion: GBR ============
[YS] GBR | CV_MAE=25.3845±1.9263 | CV_RMSE=36.7614±3.2101 | CV_R2=0.8920±0.0263

============ Stage 2 | Target: UTS | Champion: ETR ============
[UTS] ETR | CV_MAE=24.8193±2.4059 | CV_RMSE=33.6644±3.3159 | CV_R2=0.8864±0.0355

============ Stage 2 | Target: El | Champion: GBR ============
[El] GBR | CV_MAE=1.8909±0.4323 | CV_RMSE=3.0140±1.0337 | CV_R2=0.7590±0.1085

第二阶段完成。


,target,selected_model,CV_MAE_mean,CV_MAE_std,CV_RMSE_mean,CV_RMSE_std,CV_R2_mean,CV_R2_std,best_value_MAE
0,YS,GBR,25.384454,1.926324,36.761422,3.210121,0.892047,0.026321,25.384454
1,UTS,ETR,24.819343,2.405933,33.664388,3.315897,0.886447,0.035475,24.819343
2,El,GBR,1.890872,0.432320,3.014018,1.033711,0.758984,0.108484,1.890872


In [11]:
# ============================================================
# Cell 11：在全开发集上训练最终模型，并在独立测试集上验证
# ============================================================
final_models = {}
test_result_records = []
test_pred_detail = {}

for target in TARGETS:
    model_name = stage1_champions[target]
    best_params = final_best_params[target]

    df_dev_t = df_dev.dropna(subset=[target]).reset_index(drop=True)
    df_test_t = df_test.dropna(subset=[target]).reset_index(drop=True)

    X_dev = df_dev_t[FEATURES].values
    y_dev = df_dev_t[target].values
    X_test = df_test_t[FEATURES].values
    y_test = df_test_t[target].values

    final_model = build_model(
        model_name=model_name,
        params=best_params,
        input_dim=X_dev.shape[1],
        for_search=False
    )
    final_model.fit(X_dev, y_dev)
    y_test_pred = final_model.predict(X_test)

    final_models[target] = {
        'model_name': model_name,
        'model': final_model,
        'best_params': best_params
    }

    m = calc_reg_metrics(y_test, y_test_pred)
    test_result_records.append({
        'target': target,
        'selected_model': model_name,
        'test_R2': m['R2'],
        'test_MAE': m['MAE'],
        'test_RMSE': m['RMSE'],
        'n_test': int(len(y_test))
    })

    detail_df = df_test_t[FEATURES + [target]].copy()
    detail_df[f'{target}_pred'] = y_test_pred
    detail_df[f'{target}_residual'] = detail_df[target] - detail_df[f'{target}_pred']
    test_pred_detail[target] = detail_df

test_results_df = pd.DataFrame(test_result_records)
test_dir = make_output_subdir('test_results')

test_results_df.to_excel(test_dir / 'independent_test_results.xlsx', index=False)
for target, detail_df in test_pred_detail.items():
    detail_df.to_excel(test_dir / f'{target}_test_prediction_details.xlsx', index=False)

print('独立测试集验证完成。')
display(test_results_df)


独立测试集验证完成。


,target,selected_model,test_R2,test_MAE,test_RMSE,n_test
0,YS,GBR,0.948382,20.157329,27.750500,64
1,UTS,ETR,0.949372,19.077107,25.839405,64
2,El,GBR,0.852942,1.577120,2.203571,64


In [12]:
# ============================================================
# Cell 12：汇总导出，方便论文或后续分析直接使用
# ============================================================
summary_dir = make_output_subdir('summary')

with pd.ExcelWriter(summary_dir / 'all_summary_tables.xlsx') as writer:
    stage1_results_df.to_excel(writer, sheet_name='stage1_fixed_param_screening', index=False)
    champion_df.to_excel(writer, sheet_name='stage1_champions', index=False)
    stage2_results_df.to_excel(writer, sheet_name='stage2_final_tuning', index=False)
    test_results_df.to_excel(writer, sheet_name='independent_test', index=False)

print('全部关键结果已汇总导出。')
print('你现在最需要看的文件通常是：')
print('1) stage1_fixed_param_screening_results.xlsx')
print('2) stage1_champion_models.xlsx')
print('3) stage2_final_tuning_results.xlsx')
print('4) independent_test_results.xlsx')


全部关键结果已汇总导出。
你现在最需要看的文件通常是：
1) stage1_fixed_param_screening_results.xlsx
2) stage1_champion_models.xlsx
3) stage2_final_tuning_results.xlsx
4) independent_test_results.xlsx
